<img align="right" src="https://raw.githubusercontent.com/joshbeckham-lab/imagefiles/main/virtual%20cures_RGB_orange_nickname.png" width="100" height="100" alt="Virtual Cures Logo" />

NAME: **Subscores Scatter Plot**

started:  5/11/2026   revised:  8/16/26     Dr. B

Note: be sure to SAVE your notebook after changes with Ctrl+S

PURPOSE:
-


From MOLSSI proposal:
...
The next opportunity is through data visualization. Currently, students use Excel to generate graphs of their docking scores and show the relationship between the sub-scores for hydrogen bonding and the hydrophobic interactions. For this assignment, I propose having students use Python to make these graphs and include indicators to distinguish positive and negative controls on the graph along with an R2 (R-squared) value to measure correlation between the sub-scores (see Fig 1 & Fig 2).

## Overview of Steps
1. get some sample data from a GOLD docking run (e.g. VS1)
* [LabVirtualScreen1VDS](https://docs.google.com/document/d/1-FOweaiO6JEXl-wJPf9LdE7fJuFbLmZhM6HqAiSaU4k/edit?usp=sharing)
2. import libraries. (numpy, matplotlib?)
3. intake data directly from the bestranking.lst (instead of as .CSV)
4. create graph of:
  - Gold score vs. S(PLP)
  - Gold score vs. S(Hbond)
  ## TBD .......
  - bar graph of GOLD Score (Y-axis) vs. Ligand #  (i.e. 1-20)  <br>
  <br>
  


###VS1 Instructions from the protocol
Does the number of polar contacts correlate to the overall Score? If not, explain what other factors in the GOLD score are affecting the binding.  _______
Make two scatter plots of both Score vs. S(PLP) and Score vs. S(hbond) (give them Figure numbers). Show a best fit line (like you have done in Beer’s Law with the correlation).

How well do these two Sub-scores correlate to the overall Score? _______
What piece of information (a number) do you use to quantify this (be sure to show it on the graph)? _______

Is there a sharp drop off in the scores at any point?  _______
Make a bar graph of GOLD Score (Y-axis) vs. Ligand #  (i.e. 1-20)  in Rank order not number order (so highest scoring is on far end of the graph and lowest scoring is on other end.)
give it a Figure number

Is there a ‘magic number’ which implies that the ligand will be a good binder?  _______

If so, how many of the ligands do you think would be worth keeping for further experimentation? _______
Compare Ligands #2 vs. #3. And then Ligands #4 vs. #5.

These are tautomer pairs. Do the scores between #2 and #3 differ much?  _______
Do the scores between #4 and #5 vary? – Would you expect this or not? _______

How did the 4 ‘decoy’ ligands perform relative to the known binders (the 16 original ligands)?
Do they end up in the active site?  _______
Are the scores better or worse than the rest?  _______
Which component of the score accounts for most of this difference for these 4 ligands relative to the other 16 ligands - S(PLP), S(hbond),  S(cho),  S(metal), DE(clash), DE(tors), intcor, RMS (heavy),? _______

# Prompts to Gemini
Gemini was used to create this script by Dr. B and Antonia P. Here are the first few prompts and revised prompts. Many more were used to get the final product. 

6/14/26

**First Prompt to Gemini:**

"Hi, for this list of docking scores from virtual screening that I am uploading as a .txt file, could you generate python code for me that I can use in a Google Colab jupyter noteboook to create a scatter plot of the Score on the Y-axis vs. the S(PLP) sub-score on the x-axis. The graph should have a linear best fit line applied to the data and display this as a line on the graph with the equation shown and the R-squared displayed as well on the graph in a corner that does not overlap with the data. Each data point should have a small text label next to it with the Ligand name.  Please offer the user a suggested caption starting with "Fig 1."  and provide a user input where they can add more information to the caption as desired. A second plot of the Score on the Y-axis vs. the S(hbond) sub-score on the x-axis should also be generated. "

**Revisions to Gemini:**

"ok, this works well, however, for the figure caption, can we show the full caption in the user input box so they can edit or append to it directly? Also, let's show the user the first plot and ask for the caption input, then show the second plot and ask for it's caption input (instead of showing both figures first). Also, the position of some of the labels overlaps with others, they need to be relocated around the dot so that they don't overlap. Also, the position of the equation and R2 boxes should be along the top like they are but not where there is any data.  So in the S(PLP) one it could be positioned at the top right. Lastly, let's genearate .png files of these graphs in the users folder so that they can be downloaded. The captions should be generated as separate .txt files.  Then a user prompt to download them should be given."

**Revisions 2 to Gemini:**

"this worked very well. However, can we remove the figure legend.  Also, when I ran this it gave an error: <br>"
WARNING:adjustText:Looks like you are using a tranform that doesn't support FancyArrowPatch, using ax.annotate instead. The arrows might strike through texts. Increasing shrinkA in arrowprops might help."  however, it seemed to run fine. Lastly, let's keep figure one displayed as we show figure 2. "

**Revisions 3 to Gemini:**
"great!  can we make the download just download all four files at once?"  <br>
it suggests .zip <br>
"for this one, can we just make it download them as individual files and not zip?" <br>
"lastly, can we remove the title from the graphs. That info should be contained in the captions only. "

## ChemCompute / Jupyter instructions

1. Click in the cell you want to run
2. Then click on triangle 'arrow' at top to run that cell. (or, in the menu, click Run)
3. Use **Choose GOLD File** and select either a `.txt` or `.lst` bestranking file. The file picker intentionally allows all file types because ChemCompute may not recognize `.lst` as a standard extension.
4. It will generate **both graphs immediately** after upload.
5. Edit both captions and click **Save Captions & Finish**.
6. Click **Show Download Links** to access both PNG and both TXT files.


In [2]:
#@title Scatter Plots for GOLD docking results: Version 6  Gemini   8/16/26  
#(Warning Suppressed)
import io
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

# 1. Suppress cosmetic adjustText warnings
warnings.filterwarnings("ignore", message=".*FancyArrowPatch.*")
warnings.filterwarnings("ignore", module="adjustText")

try:
    from adjustText import adjust_text
except ImportError:
    os.system('pip install adjustText -q')
    from adjustText import adjust_text

# 2. Universal File Payload Handler (ipywidgets v7 and v8 compatible)
def uploaded_value_to_file(upload_value):
    if not upload_value:
        return None, None
    
    if isinstance(upload_value, (tuple, list)):
        if len(upload_value) == 0:
            return None, None
        item = upload_value[0]
        filename = item.get('name', 'uploaded.txt')
        content = item.get('content')
        if isinstance(content, memoryview):
            content = content.tobytes()
        return filename, content

    if isinstance(upload_value, dict):
        if not upload_value:
            return None, None
        filename = list(upload_value.keys())[0]
        item = upload_value[filename]
        content = item.get('content', b'')
        if isinstance(content, memoryview):
            content = content.tobytes()
        return filename, content

    return None, None

# 3. Robust GOLD bestranking Parser
def parse_bestranking_bytes(content_bytes):
    text = content_bytes.decode('utf-8', errors='ignore')
    lines = text.splitlines()

    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith('#'):
            if any(key in stripped for key in ['Score', 'S(PLP)', 'S(hbond)', 'Fitness']):
                hdr = stripped.lstrip('#').strip()
                hdr = hdr.replace('File name', 'File_name')
                hdr = hdr.replace('Ligand name', 'Ligand_name')
                hdr = hdr.replace('RMS(heavy)', 'RMS_heavy')
                cleaned_lines.append(hdr)
        else:
            cleaned_lines.append(stripped)

    clean_text = "\n".join(cleaned_lines)
    if not clean_text.strip():
        raise ValueError("Uploaded file appears empty or unreadable.")

    df = pd.read_csv(io.StringIO(clean_text), sep=r'\s+')

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip("'\"")

    if 'Score' not in df.columns and 'Fitness' in df.columns:
        df['Score'] = df['Fitness']

    required_cols = ['Score', 'S(PLP)', 'S(hbond)']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required column(s): {missing}. Found: {list(df.columns)}")

    if 'Ligand_name' not in df.columns:
        df['Ligand_name'] = [f"Ligand_{i+1}" for i in range(len(df))]

    return df

# 4. Download Utility
def show_download_links(filenames):
    for fname in filenames:
        if os.path.exists(fname):
            display(FileLink(fname))
        else:
            print(f"File not found: {fname}")

# 5. Dynamic Plotting Engine
def generate_scatter_plot_v4(df, x_col, y_col, save_filename):
    fig, ax = plt.subplots(figsize=(11, 8))
    x = df[x_col].astype(float)
    y = df[y_col].astype(float)

    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    r_squared = r_value ** 2

    ax.scatter(x, y, color='darkblue', edgecolor='k', s=75, zorder=3)
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = slope * x_fit + intercept
    ax.plot(x_fit, y_fit, color='crimson', linestyle='--', linewidth=2, zorder=2)

    x_mid = (x.min() + x.max()) / 2
    y_mid = (y.min() + y.max()) / 2
    top_left_count = ((x < x_mid) & (y > y_mid)).sum()
    top_right_count = ((x >= x_mid) & (y > y_mid)).sum()

    box_x, box_alignment = (0.05, 'left') if top_left_count <= top_right_count else (0.95, 'right')

    eq_text = f"Equation: y = {slope:.4f}x + ({intercept:.4f})\n$R^2$ = {r_squared:.4f}"
    ax.text(
        box_x, 0.96, eq_text,
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment='top',
        horizontalalignment=box_alignment,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.95),
        zorder=4
    )

    labels = []
    for i, row in df.iterrows():
        t = ax.text(
            row[x_col], row[y_col],
            str(row['Ligand_name']),
            fontsize=9.5,
            weight='bold',
            color='black'
        )
        labels.append(t)

    # Relocate names with warning suppression context and label margin buffers
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adjust_text(
            labels,
            arrowprops=dict(
                arrowstyle="->",
                color='gray',
                lw=0.6,
                alpha=0.7,
                shrinkA=3,
                shrinkB=3
            ),
            expand_points=(1.6, 1.6),
            force_points=0.2,
            ax=ax
        )

    ax.set_xlabel(x_col, fontsize=12, weight='bold')
    ax.set_ylabel(y_col, fontsize=12, weight='bold')
    ax.grid(True, linestyle=':', alpha=0.6, zorder=1)
    plt.tight_layout()

    plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    plt.show()

# 6. Dashboard Interface
class DockingAppV4:
    def __init__(self, df):
        self.df = df
        self.caption1_default = (
            "Fig 1. Correlation analysis of overall GOLD fitness Score against "
            "piece-wise linear potential sub-score S(PLP) for top-ranked virtual screening hits."
        )
        self.caption2_default = (
            "Fig 2. Correlation analysis of overall GOLD fitness Score against "
            "hydrogen bonding sub-score S(hbond) for top-ranked virtual screening hits."
        )
        self.render_all()

    def render_all(self):
        clear_output(wait=True)

        print("Showing Graphical Frame 1/2...")
        generate_scatter_plot_v4(self.df, 'S(PLP)', 'Score', 'score_vs_splp.png')

        print("\n" + "-" * 100)
        print("Showing Graphical Frame 2/2...")
        generate_scatter_plot_v4(self.df, 'S(hbond)', 'Score', 'score_vs_shbond.png')

        print("\nEdit the captions below, then click Save Captions & Finish.")

        self.ta1 = widgets.Textarea(value=self.caption1_default, description='Edit Fig 1:', layout=widgets.Layout(width='95%', height='90px'))
        self.ta2 = widgets.Textarea(value=self.caption2_default, description='Edit Fig 2:', layout=widgets.Layout(width='95%', height='90px'))
        self.save_button = widgets.Button(description='Save Captions & Finish', button_style='success', layout=widgets.Layout(width='280px'))
        self.download_button = widgets.Button(description='Show Download Links', button_style='info', layout=widgets.Layout(width='240px'), disabled=True)
        self.result_output = widgets.Output()

        self.save_button.on_click(self.save_captions)
        self.download_button.on_click(self.show_downloads)

        display(self.ta1, self.ta2, widgets.HBox([self.save_button, self.download_button]), self.result_output)

    def save_captions(self, _):
        with open('caption_splp.txt', 'w', encoding='utf-8') as f:
            f.write(self.ta1.value)
        with open('caption_shbond.txt', 'w', encoding='utf-8') as f:
            f.write(self.ta2.value)

        self.ta1.disabled = True
        self.ta2.disabled = True
        self.save_button.disabled = True
        self.save_button.description = 'Captions Saved ✓'
        self.download_button.disabled = False

        with self.result_output:
            clear_output()
            print("Both graphs and both captions were saved successfully.")
            print("Click Show Download Links to access all four files.")

    def show_downloads(self, _):
        with self.result_output:
            clear_output()
            print("Generated files:")
            show_download_links(['score_vs_splp.png', 'caption_splp.txt', 'score_vs_shbond.png', 'caption_shbond.txt'])

# 7. Execution Pipeline
def run_pipeline_v4():
    upload_widget = widgets.FileUpload(accept="", multiple=False, description="1. Choose File")
    process_button = widgets.Button(description="2. Analyze Uploaded File", button_style='primary')
    status_output = widgets.Output()

    def process_file_action(_=None):
        with status_output:
            clear_output()
            try:
                filename, content_bytes = uploaded_value_to_file(upload_widget.value)
                if not filename or not content_bytes:
                    print("⚠️ Please select your 'bestranking' file using '1. Choose File', then click '2. Analyze Uploaded File'.")
                    return
                
                df = parse_bestranking_bytes(content_bytes)
                print(f"Loaded {filename} ({len(df)} rows). Plotting data...")
                DockingAppV4(df)
            except Exception as exc:
                print(f"❌ Error processing file: {exc}")

    upload_widget.observe(process_file_action, names="value")
    process_button.on_click(process_file_action)

    display(widgets.HBox([upload_widget, process_button]), status_output)

run_pipeline_v4()

Output()

In [ ]:
# Safe cleanup for Jupyter / ChemCompute
# Removes only files generated by this notebook.

import os
from IPython.display import clear_output

generated_files = [
    "score_vs_splp.png",
    "caption_splp.txt",
    "score_vs_shbond.png",
    "caption_shbond.txt",
]

removed = []
for filename in generated_files:
    if os.path.isfile(filename):
        os.remove(filename)
        removed.append(filename)

clear_output()

if removed:
    print("Removed generated files:")
    for filename in removed:
        print(f" - {filename}")
else:
    print("No generated scatter-plot files were present.")

print(
    "\nYour notebook and uploaded GOLD files were preserved.\n"
    "Use the Jupyter menu option to clear displayed cell outputs if needed."
)
